# Lesson 2.3 · Missing data

A missing value is not an error to eliminate: it is information about how the data were collected, and sometimes it is
the most important information. This lesson distinguishes the missingness mechanisms, shows the scikit-learn imputers and
explains why the choice of imputation changes the distribution of the data.

**What you will learn**

- The three mechanisms: MCAR, MAR and MNAR, and what can be inferred from the data.
- `SimpleImputer`, `KNNImputer`, `IterativeImputer` and the missingness indicator.
- Why imputation with the mean shortens the variance and weakens correlations.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.experimental import enable_iterative_imputer          # noqa: F401  (needed to import it)
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer

df = ds.loans()
df.isna().mean().round(3)

## Three mechanisms

| Mechanism | Meaning | Example |
|---|---|---|
| **MCAR** (Missing Completely At Random) | the missingness depends on nothing | a sensor that loses a reading at random |
| **MAR** (Missing At Random) | it depends on other **observed** variables | someone who is unemployed does not report their job tenure |
| **MNAR** (Missing Not At Random) | it depends on the **missing** value itself | someone who earns a lot does not declare their income |

From a sample you can **detect** dependencies on the observed variables (MAR), but you **cannot prove MCAR nor
distinguish MNAR**: the missing value, by definition, we do not see.

In [ ]:
missing = df["income"].isna()
print("default rate with income present :", round(df.loc[~missing, "defaulted"].mean(), 3))
print("default rate with income missing :", round(df.loc[missing, "defaulted"].mean(), 3))

The fact that income is missing is **predictive**: whoever does not declare it defaults much more often. Imputing the value and
forgetting that it was missing throws this information away. That is why `SimpleImputer(add_indicator=True)` adds a
binary "was missing" column for every column with missing values.

## The imputers

- `SimpleImputer(strategy="mean" | "median" | "most_frequent" | "constant")`: one value per column, fast and robust.
- `KNNImputer(n_neighbors=5)`: the mean of the values of the most similar neighbors; it requires scaled features.
- `IterativeImputer`: it models each feature as a function of the others and iterates; it captures the correlations, but it is slower and less stable.

In [ ]:
num = df[["age", "income", "job_tenure", "amount"]]
for name, imp in [("mean", SimpleImputer(strategy="mean")), ("median", SimpleImputer(strategy="median")),
                  ("with indicator", SimpleImputer(strategy="median", add_indicator=True)),
                  ("KNN", KNNImputer(n_neighbors=5))]:
    Z = imp.fit_transform(num)
    print(f"{name:15s} shape {Z.shape} | remaining nan {int(np.isnan(Z).sum())}")

## What imputation with the mean does

Replacing the missing values with a constant **shortens the variance** and **weakens the correlations** with the other variables:
all the imputed values are equal, so they do not vary together with anything. If a fraction p of the values is
imputed with the mean, the variance drops by about a factor (1 − p).

In [ ]:
rng = np.random.default_rng(0)
x = rng.normal(size=5000)
for p in [0.1, 0.3, 0.5]:
    x_miss = np.where(rng.random(5000) < p, np.nan, x)
    x_imp = SimpleImputer(strategy="mean").fit_transform(x_miss.reshape(-1, 1)).ravel()
    print(f"imputed share {p:.0%}: variance {x_imp.var():.2f} (true 1.00, expected {1 - p:.2f})")

> **Common mistake.** Imputing with the mean (or dropping the incomplete rows) without asking **why** the value is missing: if the missingness is informative, imputation erases it, and the mean reduces variance and correlations (exercises C3 and D2). You impute inside the pipeline, adding the missingness indicator when it matters.

## Check your understanding

> *Click on a question to show its answer.*

<details><summary>I drop all the rows with at least one missing value. Is that fine?</summary>

Only with MCAR and with very few rows lost. In the loans dataset about 29% of the rows have at least one missing value, and the dropped rows are not a random sample: whoever does not declare income defaults much more often. The model would see a population different from the one it will be used on.

</details>

<details><summary>When do I fit the imputer?</summary>

Only on the training set, like any other transformer: the mean (or the model of the neighbors) is a statistic learned from the data. Fitting it on the whole dataset lets information from the test set into the training.

</details>